# Pipeline final KNN

Notebook ini menjelaskan alur KNN dari pembacaan data mentah sampai pembuatan berkas prediksi. Implementasi model dan prapemrosesan menggunakan modul bersama di `src/experiments/knn` agar kode eksperimen tidak diduplikasi.

Konfigurasi terpilih menggunakan 21 tetangga, bobot berdasarkan jarak, metrik Manhattan, bobot kategori 1.0, dan ambang 0.5. Macro F1 validasi yang tercatat sebesar 0.841689.

## Rancangan evaluasi

Data validasi dibuat dengan seed 42 dan mengambil sekitar 20 persen dari tiap kelas. Indeks split yang sama dipakai untuk seluruh evaluasi. Statistik prapemrosesan hanya dipelajari dari bagian latih.

Target `is_canceled` tidak digunakan sebagai fitur dan `id` hanya dipakai untuk menjaga identitas baris pada submission. Data test tidak memiliki label, sehingga tidak digunakan untuk pemilihan konfigurasi maupun evaluasi.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.neighbors import KNeighborsClassifier

ROOT_DIR = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "train.csv").is_file()),
    None,
)
if ROOT_DIR is None:
    raise FileNotFoundError("Jalankan notebook dari direktori proyek")
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.experiments.knn.knn import KNNClassifier
from src.experiments.knn.preprocessing import KNNPreprocessor
from src.experiments.knn.run_cost_sensitive_experiment import evaluate_cost_sensitive
from src.experiments.knn.run_experiments import make_split

SEED = 42
BATCH_SIZE = 2000
FINAL_CONFIG = {
    "n_neighbors": 21,
    "weights": "distance",
    "metric": "manhattan",
    "cat_weight": 1.0,
    "threshold": 0.5,
}
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"
KNN_OUTPUT_DIR = OUTPUT_DIR / "knn"
MODEL_DIR = OUTPUT_DIR / "final" / "model" / "knn"

print(f"NumPy {np.__version__} dan pandas {pd.__version__}")

## Data dan split validasi

Pemeriksaan awal memastikan kolom identitas dan target tersedia, identitas baris unik, target biner, serta fitur train dan test memiliki urutan kolom yang sama.

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
test_df = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})

required_train = {"id", "is_canceled"}
if not required_train.issubset(train_df.columns) or "id" not in test_df.columns:
    raise ValueError("Kolom id atau target tidak ditemukan")
if not train_df["id"].is_unique or not test_df["id"].is_unique:
    raise ValueError("Kolom id harus unik pada train dan test")

feature_columns = [
    column for column in train_df.columns if column not in {"id", "is_canceled"}
]
if feature_columns != [column for column in test_df.columns if column != "id"]:
    raise ValueError("Fitur train dan test tidak cocok atau berbeda urutan")

y = train_df["is_canceled"].to_numpy(dtype=np.int64)
if not np.isin(y, [0, 1]).all():
    raise ValueError("Target hanya boleh berisi label 0 dan 1")
train_idx, valid_idx = make_split(y)
X_raw = train_df[feature_columns].copy()
X_test_raw = test_df[feature_columns].copy()
y_train, y_valid = y[train_idx], y[valid_idx]

print(f"Train berisi {len(train_idx)} baris")
print(f"Validasi berisi {len(valid_idx)} baris")
display(pd.DataFrame({
    "train": pd.Series(y_train).value_counts().reindex([0, 1], fill_value=0),
    "validation": pd.Series(y_valid).value_counts().reindex([0, 1], fill_value=0),
}))

In [ ]:
feature_profile = pd.DataFrame({
    "dtype": X_raw.dtypes.astype(str),
    "missing_train": X_raw.iloc[train_idx].isna().sum(),
    "missing_validation": X_raw.iloc[valid_idx].isna().sum(),
    "unique_train": X_raw.iloc[train_idx].nunique(dropna=True),
})
display(feature_profile)
print(f"Jumlah fitur mentah {len(feature_columns)}")
print(f"Jumlah baris train {len(train_df)} dan test {len(test_df)}")

## Prapemrosesan dan model

Fitur numerik selain dua hitungan riwayat dibakukan dengan rata-rata dan simpangan baku data latih. `prior_cancellations` dan `prior_completed_bookings` diubah menjadi indikator di atas median latih. Fitur kategorikal diubah menjadi one-hot, termasuk kolom untuk nilai hilang dan kategori yang belum pernah terlihat.

Prapemroses hanya di-fit pada bagian latih. Transformasi validasi memakai statistik dan kategori yang sama. Bobot kategori mengatur kontribusi blok kategorikal terhadap jarak.

In [ ]:
preprocessor = KNNPreprocessor(
    cat_weight=FINAL_CONFIG["cat_weight"]
).fit(X_raw.iloc[train_idx])
X_train_model = preprocessor.transform(X_raw.iloc[train_idx])
X_valid_model = preprocessor.transform(X_raw.iloc[valid_idx])

model = KNNClassifier(
    n_neighbors=FINAL_CONFIG["n_neighbors"],
    weights=FINAL_CONFIG["weights"],
    metric=FINAL_CONFIG["metric"],
    threshold=FINAL_CONFIG["threshold"],
).fit(X_train_model, y_train)
valid_scores = model.decision_scores(X_valid_model, batch_size=BATCH_SIZE)
valid_predictions = (valid_scores > FINAL_CONFIG["threshold"]).astype(np.int64)
if valid_predictions.shape != y_valid.shape:
    raise ValueError("Prediksi validasi tidak memiliki satu label per baris")

validation_macro_f1 = f1_score(
    y_valid, valid_predictions, labels=[0, 1], average="macro", zero_division=0
)
validation_matrix = confusion_matrix(y_valid, valid_predictions, labels=[0, 1])
precision, recall, class_f1, support = precision_recall_fscore_support(
    y_valid, valid_predictions, labels=[0, 1], zero_division=0
)
validation_metrics = pd.DataFrame({
    "precision": precision,
    "recall": recall,
    "f1": class_f1,
    "support": support,
}, index=pd.Index([0, 1], name="kelas"))
print(f"Macro F1 validasi {validation_macro_f1:.6f}")
display(validation_metrics)
display(pd.DataFrame(
    validation_matrix,
    index=pd.Index([0, 1], name="aktual"),
    columns=pd.Index([0, 1], name="prediksi"),
))

### Insight validasi

Macro F1 0.841689 menjadi acuan konfigurasi final. Pada matriks validasi tercatat 290 false positive dan 455 false negative. Precision kelas batal sebesar 0.8220 dan recall kelas batal sebesar 0.7464. Skor merangkum keseimbangan kedua kelas, sedangkan matriks memperlihatkan jenis kesalahan yang masih terjadi.

## Pembanding scikit-learn

Pembanding memakai fitur hasil prapemrosesan yang sama, indeks split yang sama, dan parameter tetangga yang sama. Implementasi dari nol tetap menjadi model yang dievaluasi untuk pipeline ini. Perbedaan kecil dapat timbul dari aturan pemilihan tetangga ketika jarak sama.

In [ ]:
sklearn_model = KNeighborsClassifier(
    n_neighbors=FINAL_CONFIG["n_neighbors"],
    weights=FINAL_CONFIG["weights"],
    metric=FINAL_CONFIG["metric"],
    n_jobs=-1,
).fit(X_train_model, y_train)
sklearn_scores = sklearn_model.predict_proba(X_valid_model)[:, 1]
sklearn_predictions = (sklearn_scores > FINAL_CONFIG["threshold"]).astype(np.int64)
sklearn_macro_f1 = f1_score(
    y_valid, sklearn_predictions, labels=[0, 1], average="macro", zero_division=0
)
display(pd.DataFrame({
    "model": ["KNN dari nol", "scikit-learn"],
    "macro_f1": [validation_macro_f1, sklearn_macro_f1],
    "beda_prediksi": [
        0,
        int(np.count_nonzero(valid_predictions != sklearn_predictions)),
    ],
}))

## Ringkasan eksperimen

Grid nilai k dan pencarian ambang tidak melampaui batas adopsi 0.005, sehingga nilai awal k 21 dan ambang 0.5 dipertahankan. Penggantian metrik Euclidean menjadi Manhattan menaikkan Macro F1 dari 0.831783 menjadi 0.841689. Kenaikan sebesar 0.009906 melewati batas adopsi.

Bobot kategori 0.5 menghasilkan skor 0.843450, tetapi kenaikan 0.001761 belum melewati batas adopsi. Bobot 1.0 tetap dipakai. Seluruh kelompok fitur turunan yang diuji menurunkan skor. `total_guests` mencatat 0.839597 dan `total_stay_value` mencatat 0.832653, sehingga fitur dasar dipertahankan.

Hasil ini membandingkan satu perubahan pada satu tahap dari proses seleksi yang dilakukan. Skor validasi dipakai untuk memilih konfigurasi dan tidak menjadi estimasi evaluasi independen.

## Analisis biaya kesalahan

Analisis tambahan memberi biaya dua untuk false positive dan satu untuk false negative. Ambang teoretis 2/3 berlaku jika skor dapat ditafsirkan sebagai probabilitas terkalibrasi. Skor voting KNN belum tentu memenuhi asumsi tersebut, sehingga ambang dipilih secara empiris pada validasi dan hasilnya dibaca sebagai analisis sensitivitas.

In [ ]:
cost_results, cost_comparison = evaluate_cost_sensitive(
    y_valid, valid_scores, KNN_OUTPUT_DIR
)
display(cost_comparison[[
    "strategy", "threshold", "fp", "fn", "cost", "cost_per_booking", "macro_f1"
]])

Ambang 0.60 menurunkan biaya validasi dari 1035 menjadi 939, atau 9.28 persen. False positive turun dari 290 menjadi 158, sedangkan false negative naik dari 455 menjadi 623. Macro F1 turun dari 0.841689 menjadi 0.825069. Karena ambang dipilih dan dinilai pada validasi yang sama, hasil ini dapat optimistis dan tidak menggantikan konfigurasi utama.

## Fit ulang dan pembuatan submission

Setelah konfigurasi ditetapkan, prapemroses dan model di-fit ulang menggunakan seluruh data train. Prediksi test mengikuti urutan `id` pada berkas sumber. Berkas model menyimpan data latih terproses, label, konfigurasi, dan state prapemrosesan agar prediksi dapat direkonstruksi.

In [ ]:
final_preprocessor = KNNPreprocessor(
    cat_weight=FINAL_CONFIG["cat_weight"]
).fit(X_raw)
X_full = final_preprocessor.transform(X_raw)
X_test = final_preprocessor.transform(X_test_raw)
final_model = KNNClassifier(
    n_neighbors=FINAL_CONFIG["n_neighbors"],
    weights=FINAL_CONFIG["weights"],
    metric=FINAL_CONFIG["metric"],
    threshold=FINAL_CONFIG["threshold"],
).fit(X_full, y)
test_predictions = final_model.predict(X_test, batch_size=BATCH_SIZE)
if len(test_predictions) != len(test_df) or not np.isin(test_predictions, [0, 1]).all():
    raise ValueError("Prediksi test harus berisi satu label biner per baris")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    MODEL_DIR / "knn_model.npz",
    X_train=final_model.X_train,
    y_train=final_model.y_train,
)
(MODEL_DIR / "preprocessing.json").write_text(
    json.dumps(final_preprocessor.to_dict(), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
model_config = {
    "algorithm": "KNN from scratch",
    "k": FINAL_CONFIG["n_neighbors"],
    "weights": FINAL_CONFIG["weights"],
    "metric": FINAL_CONFIG["metric"],
    "cat_weight": FINAL_CONFIG["cat_weight"],
    "threshold": FINAL_CONFIG["threshold"],
    "seed": SEED,
    "macro_f1_validasi": round(float(validation_macro_f1), 6),
    "feature_engineering": "tidak ada fitur turunan yang diadopsi",
}
(MODEL_DIR / "konfigurasi.json").write_text(
    json.dumps(model_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "is_canceled": test_predictions.astype(np.int64),
})
if submission.columns.tolist() != ["id", "is_canceled"]:
    raise ValueError("Kolom submission tidak sesuai kontrak")
if not submission["id"].equals(test_df["id"].reset_index(drop=True)):
    raise ValueError("Urutan id submission berubah")
KNN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_path = KNN_OUTPUT_DIR / "submission_knn.csv"
submission.to_csv(submission_path, index=False)
print(f"Model tersimpan di {MODEL_DIR}")
print(f"Submission tersimpan di {submission_path} dengan {len(submission)} baris")
display(submission["is_canceled"].value_counts().sort_index().rename("jumlah"))

In [ ]:
saved_arrays = np.load(MODEL_DIR / "knn_model.npz")
saved_preprocessor = KNNPreprocessor.from_dict(json.loads(
    (MODEL_DIR / "preprocessing.json").read_text(encoding="utf-8")
))
saved_config = json.loads(
    (MODEL_DIR / "konfigurasi.json").read_text(encoding="utf-8")
)
reloaded_model = KNNClassifier(
    n_neighbors=saved_config["k"],
    weights=saved_config["weights"],
    metric=saved_config["metric"],
    threshold=saved_config["threshold"],
).fit(saved_arrays["X_train"], saved_arrays["y_train"])
reloaded_predictions = reloaded_model.predict(
    saved_preprocessor.transform(X_test_raw), batch_size=BATCH_SIZE
)
np.testing.assert_array_equal(reloaded_predictions, test_predictions)
print("Pemeriksaan muat ulang berhasil dan prediksi tetap identik")

## Analisis kesalahan validasi

Kesalahan berikut berasal dari split validasi dan dapat dipakai untuk merumuskan eksperimen lanjutan. Label validasi tidak dimasukkan kembali ke model setelah konfigurasi dipilih. Data test tidak ditampilkan karena labelnya tidak tersedia.

In [ ]:
error_analysis = X_raw.iloc[valid_idx].reset_index(drop=True).copy()
error_analysis["actual"] = y_valid
error_analysis["predicted"] = valid_predictions
error_analysis["error_type"] = np.select(
    [
        error_analysis["actual"].eq(0) & error_analysis["predicted"].eq(1),
        error_analysis["actual"].eq(1) & error_analysis["predicted"].eq(0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)
display(error_analysis["error_type"].value_counts())
display(error_analysis.loc[error_analysis["error_type"] != "correct"].head(20))

## Perbandingan biaya lintas model

Analisis awal di atas mengilustrasikan FP dua kali FN pada validasi yang sama. Eksperimen baru menggunakan FP tiga kali FN dan memilih ambang melalui tiga lipatan data latih, lalu mengukur hasilnya pada holdout. Jalankan `python -m src.experiments.knn.run_cost_sensitive_experiment` untuk tabel yang sebanding dengan model lain.

In [ ]:
cost_path = REPO_ROOT / 'output/knn/cost_sensitive_comparison.csv'
if cost_path.exists():
    display(pd.read_csv(cost_path))
else:
    print('Jalankan eksperimen biaya KNN terlebih dahulu.')